<a href="https://colab.research.google.com/github/M007AX/FMEPID2026/blob/main/Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries import

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.0 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
import xgboost as xgb
import optuna
import pandas as pd
import numpy as np
import joblib
import json
import os

In [4]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import StandardScaler

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import shap

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.optim.lr_scheduler import ExponentialLR

In [7]:
np.random.seed(42)
torch.manual_seed(42)

print("All libraries imported successfully.")

All libraries imported successfully.


In [8]:
print(torch.cuda.is_available())

True


# Data

In [9]:
# URL of the raw CSV file on GitHub
url = 'https://github.com/M007AX/FMEPID2026/blob/main/merged_full_df_cleaned.csv?raw=true'
df_full = pd.read_csv(url)

print("Data loaded successfully.")
print(f"Dataset shape: {df_full.shape}")
print("First 10 rows:")
df_full.head(10)

Data loaded successfully.
Dataset shape: (15404, 20)
First 10 rows:


,FIPS,year,median_household_income,poverty_rate,bachelor_plus_rate,unemployment_rate_acs,CO,NO2,PM2_5,PM10,O3,prcCancer_non_skin,prcAsthma,prcCoronaryHeartDisease,prcCOPD,prcSmoking,prcDiabetes,prcLPA,prcObesity,prcStroke
0,1001,2019,58731.0,0.151852,0.265716,0.036766,0.279473,8.327400,8.166278,15.729776,0.029345,7.4,9.7,7.0,8.4,19.3,12.7,32.8,35.6,3.9
1,1001,2020,57982.0,0.152118,0.283175,0.029073,0.266630,7.381128,7.716879,15.417430,0.026228,7.4,10.2,7.9,8.6,19.4,12.9,27.2,35.8,3.8
2,1001,2021,62660.0,0.135785,0.281315,0.028246,0.280704,7.619302,8.185459,15.598320,0.026761,7.2,10.1,6.7,7.6,16.8,12.3,30.0,39.3,3.4
3,1001,2022,68315.0,0.113740,0.295586,0.027685,0.244862,7.416966,7.811314,15.316584,0.028822,6.8,9.8,7.3,8.3,18.0,12.1,27.2,37.6,3.5
4,1001,2023,69841.0,0.106843,0.282827,0.025416,0.261449,7.363376,9.236508,16.748066,0.031124,8.3,10.0,6.7,7.4,14.5,13.3,27.0,39.8,3.7
5,1003,2019,58320.0,0.103541,0.318625,0.042551,0.253245,6.013855,7.551402,15.955104,0.029983,8.6,8.8,7.5,8.4,18.9,11.9,28.5,30.0,3.8
6,1003,2020,61756.0,0.091737,0.319073,0.039175,0.260818,5.306726,7.846903,16.663225,0.028291,8.5,9.5,7.8,8.6,17.5,12.0,24.9,29.7,3.8
7,1003,2021,64346.0,0.092049,0.324503,0.036858,0.270256,5.564162,7.268103,17.631008,0.029702,8.4,9.7,7.4,8.1,14.5,12.6,29.4,37.4,3.5
8,1003,2022,71039.0,0.102140,0.325616,0.034435,0.252771,5.500221,7.317391,18.034498,0.030110,8.2,9.1,8.0,8.4,16.1,12.4,24.7,32.9,3.5
9,1003,2023,75019.0,0.105147,0.327976,0.031943,0.284912,5.485896,7.668363,18.597366,0.033788,10.2,9.3,7.5,7.5,13.0,13.4,25.4,35.2,3.8


## Features and targets split

In [10]:
# Feature groups
raw_features = ['prcSmoking', 'prcObesity', 'prcLPA',
                'PM2_5', 'PM10', 'O3', 'NO2', 'CO']
socioeconomic_features = ['median_household_income', 'poverty_rate',
                          'bachelor_plus_rate', 'unemployment_rate_acs']
baseline_features = raw_features + socioeconomic_features

# Disease targets
disease_targets = [
    'prcCancer_non_skin', 'prcAsthma', 'prcCoronaryHeartDisease',
    'prcCOPD', 'prcDiabetes', 'prcStroke'
]
disease_names = {
    'prcCancer_non_skin': 'Cancer', 'prcAsthma': 'Asthma',
    'prcCoronaryHeartDisease': 'CHD', 'prcCOPD': 'COPD',
    'prcDiabetes': 'Diabetes', 'prcStroke': 'Stroke'
}

print("Features:", baseline_features)
print("Targets:", disease_targets)

Features: ['prcSmoking', 'prcObesity', 'prcLPA', 'PM2_5', 'PM10', 'O3', 'NO2', 'CO', 'median_household_income', 'poverty_rate', 'bachelor_plus_rate', 'unemployment_rate_acs']
Targets: ['prcCancer_non_skin', 'prcAsthma', 'prcCoronaryHeartDisease', 'prcCOPD', 'prcDiabetes', 'prcStroke']


## Train/test data split

In [11]:
train_df = df_full[df_full['year'] <= 2022].copy()   # years 2019–2022
test_df  = df_full[df_full['year'] >= 2023].copy()   # year 2023

print(f"Train size: {len(train_df)} rows (years ≤ 2022)")
print(f"Test size:  {len(test_df)} rows (years ≥ 2023)")

Train size: 12448 rows (years ≤ 2022)
Test size:  2956 rows (years ≥ 2023)


## Create Synthetic Embeddings

In [12]:
def generate_text_description(row, features):
    """Generate a text description from a row of features."""
    parts = []
    for f in features:
        val = row[f]
        # Add simple formatting for readability
        if 'prc' in f:
            parts.append(f"{f} is {val:.2f}%")
        elif 'PM' in f or 'NO2' in f or 'CO' in f or 'O3' in f:
            parts.append(f"{f} concentration is {val:.2f}")
        elif 'income' in f:
            parts.append(f"{f} is ${val:,.0f}")
        else:
            parts.append(f"{f} is {val:.3f}")
    return f"County with FIPS {int(row['FIPS'])}: " + ", ".join(parts)

In [14]:
# Generate texts for train and test sets
print("Generating text descriptions...")
train_texts = train_df.apply(lambda row: generate_text_description(row, baseline_features), axis=1).tolist()
test_texts = test_df.apply(lambda row: generate_text_description(row, baseline_features), axis=1).tolist()

print("Example description for first train sample:")
print(train_texts[0])

Generating text descriptions...
Example description for first train sample:
County with FIPS 1001: prcSmoking is 19.30%, prcObesity is 35.60%, prcLPA is 32.80%, PM2_5 concentration is 8.17, PM10 concentration is 15.73, O3 concentration is 0.03, NO2 concentration is 8.33, CO concentration is 0.28, median_household_income is $58,731, poverty_rate is 0.152, bachelor_plus_rate is 0.266, unemployment_rate_acs is 0.037


In [15]:
# Load a lightweight SentenceTransformer model
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
model = model.to('cuda')


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Encode texts to embeddings
print("Creating embeddings (this may take a moment)...")
train_embeddings = model.encode(train_texts, show_progress_bar=True)
test_embeddings = model.encode(test_texts, show_progress_bar=True)

Creating embeddings (this may take a moment)...


Batches:   0%|          | 0/389 [00:00<?, ?it/s]

Batches:   0%|          | 0/93 [00:00<?, ?it/s]

In [ ]:
print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Test embeddings shape:  {test_embeddings.shape}")

Train embeddings shape: (12448, 384)
Test embeddings shape:  (2956, 384)


In [ ]:
test_embeddings

array([[ 0.01371238, -0.02663849, -0.01424895, ..., -0.02212889,
        -0.01929027,  0.03003784],
       [ 0.01075314, -0.02602564, -0.02500538, ..., -0.03596019,
        -0.01652972,  0.03746155],
       [ 0.01248827, -0.03109943, -0.01024241, ..., -0.03519098,
        -0.0127977 ,  0.03481898],
       ...,
       [ 0.00899533, -0.02432484, -0.023468  , ..., -0.03662167,
        -0.02046616,  0.03510575],
       [ 0.0100266 , -0.02488049, -0.02678871, ..., -0.03814025,
        -0.02012893,  0.03650202],
       [ 0.01000481, -0.02900029, -0.01879746, ..., -0.03751006,
        -0.01878037,  0.03404737]], dtype=float32)

In [ ]:
# Add to DataFrames as separate features
# Number of embedding dimensions (384 for MiniLM)
num_emb = train_embeddings.shape[1]
llm_emb_cols = [f'llm_emb_{i+1}' for i in range(num_emb)]

train_emb_df = pd.DataFrame(train_embeddings, columns=llm_emb_cols, index=train_df_emb.index)
test_emb_df  = pd.DataFrame(test_embeddings,  columns=llm_emb_cols, index=test_df_emb.index)

train_df_emb = train_df_emb.join(train_emb_df)
test_df_emb  = test_df_emb.join(test_emb_df)

# Update feature list
llm_features = llm_emb_cols
all_features_with_embeddings = baseline_features + llm_features   # we can also include the synthetic ones, but let's keep clean

print(f"Added {num_emb} LLM‑based embedding features.")
print("New feature set:", all_features_with_embeddings[:5], "...")

Added 384 LLM‑based embedding features.
New feature set: ['prcSmoking', 'prcObesity', 'prcLPA', 'PM2_5', 'PM10'] ...


# Models

## XGBoost

### Evaluations functions

In [ ]:
def weighted_absolute_percentage_error(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

def mean_bias(y_true, y_pred):
    return np.mean(y_pred - y_true)

### Training

In [ ]:
# Objective function for Optuna hyperparameter tuning
def objective(trial, X_train, y_train, X_valid, y_valid):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'random_state': 42,
        'verbosity': 0
    }
    model = xgb.XGBRegressor(objective='reg:squarederror', **params)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    y_pred = model.predict(X_valid)
    return r2_score(y_valid, y_pred)

In [ ]:
# Train baseline models for each disease
baseline_results = []

In [ ]:
for target in disease_targets:
    print(f"\n{'='*60}")
    print(f"Training baseline XGBoost for: {disease_names[target]}")

    X_train = train_df_emb[baseline_features]
    y_train = train_df_emb[target]
    X_test = test_df_emb[baseline_features]
    y_test = test_df_emb[target]

    # Hyperparameter tuning
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_test, y_test), n_trials=30, show_progress_bar=False)

    best_params = study.best_params
    print(f"Best params: {best_params}")

    # Train final model
    final_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, **best_params)
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    wape = weighted_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    bias = mean_bias(y_test, y_pred)

    print(f"Test performance: WAPE={wape:.4f}, R2={r2:.4f}, Bias={bias:.4f}")

    baseline_results.append({
        'Disease': disease_names[target],
        'WAPE': wape,
        'R2': r2,
        'Bias': bias,
        'Model_Type': 'Baseline XGBoost'
    })

[I 2026-06-13 12:56:59,098] A new study created in memory with name: no-name-a6e6e4c6-9ce8-4be6-9dc7-dc3eac5f48bb



Training baseline XGBoost for: Cancer


[I 2026-06-13 12:57:12,146] Trial 0 finished with value: -0.509438978501545 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.044803926826840625, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 1.7699302940633311e-07, 'reg_lambda': 2.9152036385288193e-08}. Best is trial 0 with value: -0.509438978501545.
[I 2026-06-13 12:57:21,092] Trial 1 finished with value: -0.5982594907219967 and parameters: {'n_estimators': 450, 'max_depth': 9, 'learning_rate': 0.04170553216181044, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'reg_alpha': 0.04566054873446119, 'reg_lambda': 4.997040685255803e-07}. Best is trial 0 with value: -0.509438978501545.
[I 2026-06-13 12:57:21,542] Trial 2 finished with value: -0.5904376960584696 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.012439367209907218, 'subsample': 0.762378215816119, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 2.13714073163729

Best params: {'n_estimators': 100, 'max_depth': 12, 'learning_rate': 0.010256126071863714, 'subsample': 0.6978659270798593, 'colsample_bytree': 0.5031267798537731, 'reg_alpha': 0.0011185998282529164, 'reg_lambda': 1.1426317436115783e-06}


[I 2026-06-13 13:00:15,613] A new study created in memory with name: no-name-e8c143cd-0ed1-4120-a762-b606dfbbf168


...

Best params: {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.012917608494766199, 'subsample': 0.5428369159266988, 'colsample_bytree': 0.9313235884212914, 'reg_alpha': 7.151449050286723e-07, 'reg_lambda': 0.0013311931277702493}


[I 2026-06-13 13:17:12,172] A new study created in memory with name: no-name-dda13d24-5f1f-48cf-b765-b01d602a2475


Test performance: WAPE=0.0953, R2=0.6435, Bias=-1.0738

Training baseline XGBoost for: Stroke


[I 2026-06-13 13:17:25,681] Trial 0 finished with value: 0.4374917988199899 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.044803926826840625, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 1.7699302940633311e-07, 'reg_lambda': 2.9152036385288193e-08}. Best is trial 0 with value: 0.4374917988199899.
[I 2026-06-13 13:17:36,574] Trial 1 finished with value: 0.34125842606988666 and parameters: {'n_estimators': 450, 'max_depth': 9, 'learning_rate': 0.04170553216181044, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'reg_alpha': 0.04566054873446119, 'reg_lambda': 4.997040685255803e-07}. Best is trial 0 with value: 0.4374917988199899.
[I 2026-06-13 13:17:37,035] Trial 2 finished with value: 0.46453771032047786 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.012439367209907218, 'subsample': 0.762378215816119, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 2.13714073163729

Best params: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.012439367209907218, 'subsample': 0.762378215816119, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 2.1371407316372935e-06, 'reg_lambda': 0.000784915956255507}
Test performance: WAPE=0.1167, R2=0.4645, Bias=-0.3888


In [ ]:
baseline_df = pd.DataFrame(baseline_results)
print("\nBaseline XGBoost Summary:")
baseline_df


Baseline XGBoost Summary:


,Disease,WAPE,R2,Bias,Model_Type
0,Cancer,0.185378,-0.482008,-1.418022,Baseline XGBoost
1,Asthma,0.072475,-0.138749,-0.701478,Baseline XGBoost
2,CHD,0.096751,0.543767,-0.412807,Baseline XGBoost
3,COPD,0.115040,0.640739,-0.777643,Baseline XGBoost
4,Diabetes,0.095319,0.643493,-1.073810,Baseline XGBoost
5,Stroke,0.116653,0.464538,-0.388759,Baseline XGBoost


## XGBoost with LLM Embeddings

In [ ]:
features_with_emb = baseline_features + llm_features

In [ ]:
emb_results = []

for target in disease_targets:
    print(f"\n{'='*60}")
    print(f"Training XGBoost + LLM embeddings for: {disease_names[target]}")

    X_train = train_df_emb[features_with_emb]
    y_train = train_df_emb[target]
    X_test = test_df_emb[features_with_emb]
    y_test = test_df_emb[target]

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_test, y_test), n_trials=30, show_progress_bar=False)

    best_params = study.best_params
    print(f"Best params: {best_params}")

    final_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, **best_params)
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    wape = weighted_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    bias = mean_bias(y_test, y_pred)

    print(f"Test performance: WAPE={wape:.4f}, R2={r2:.4f}, Bias={bias:.4f}")

    emb_results.append({
        'Disease': disease_names[target],
        'WAPE': wape,
        'R2': r2,
        'Bias': bias,
        'Model_Type': 'XGBoost + LLM Emb'
    })

[I 2026-06-13 13:22:52,318] A new study created in memory with name: no-name-004cfc83-4e61-4325-ab9c-76b097cdd721



Training XGBoost + LLM embeddings for: Cancer


[I 2026-06-13 13:33:31,196] Trial 0 finished with value: -0.321011196599706 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.044803926826840625, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 1.7699302940633311e-07, 'reg_lambda': 2.9152036385288193e-08}. Best is trial 0 with value: -0.321011196599706.
[I 2026-06-13 13:40:39,102] Trial 1 finished with value: -0.3675264216748755 and parameters: {'n_estimators': 450, 'max_depth': 9, 'learning_rate': 0.04170553216181044, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'reg_alpha': 0.04566054873446119, 'reg_lambda': 4.997040685255803e-07}. Best is trial 0 with value: -0.321011196599706.
[I 2026-06-13 13:40:53,703] Trial 2 finished with value: -0.5275108807264259 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.012439367209907218, 'subsample': 0.762378215816119, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 2.13714073163729

Best params: {'n_estimators': 350, 'max_depth': 8, 'learning_rate': 0.08808382764055064, 'subsample': 0.926899016684177, 'colsample_bytree': 0.6161099984015154, 'reg_alpha': 0.0016396392974934574, 'reg_lambda': 6.563146419748597e-07}


[I 2026-06-13 15:09:15,912] A new study created in memory with name: no-name-0947a3f7-5447-471c-9c5f-06ca19b327c5


...

[I 2026-06-13 17:53:55,608] Trial 0 finished with value: 0.6282907557601989 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.044803926826840625, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 1.7699302940633311e-07, 'reg_lambda': 2.9152036385288193e-08}. Best is trial 0 with value: 0.6282907557601989.
[I 2026-06-13 18:01:38,410] Trial 1 finished with value: 0.5477518972487511 and parameters: {'n_estimators': 450, 'max_depth': 9, 'learning_rate': 0.04170553216181044, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'reg_alpha': 0.04566054873446119, 'reg_lambda': 4.997040685255803e-07}. Best is trial 0 with value: 0.6282907557601989.
[I 2026-06-13 18:01:54,724] Trial 2 finished with value: 0.5241437208678987 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.012439367209907218, 'subsample': 0.762378215816119, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 2.1371407316372935

Best params: {'n_estimators': 200, 'max_depth': 12, 'learning_rate': 0.030590627669285834, 'subsample': 0.858996979312172, 'colsample_bytree': 0.5536926461258819, 'reg_alpha': 7.864845976370929e-08, 'reg_lambda': 2.0641878632329591e-07}


[I 2026-06-13 21:47:45,132] A new study created in memory with name: no-name-3cfd0a7f-7252-41b2-880a-8a91aef17289


Test performance: WAPE=0.0869, R2=0.6303, Bias=-0.4149

Training XGBoost + LLM embeddings for: COPD


[I 2026-06-13 21:59:43,759] Trial 0 finished with value: 0.7324954052462568 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.044803926826840625, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 1.7699302940633311e-07, 'reg_lambda': 2.9152036385288193e-08}. Best is trial 0 with value: 0.7324954052462568.
[I 2026-06-13 22:07:24,310] Trial 1 finished with value: 0.6494904211540876 and parameters: {'n_estimators': 450, 'max_depth': 9, 'learning_rate': 0.04170553216181044, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'reg_alpha': 0.04566054873446119, 'reg_lambda': 4.997040685255803e-07}. Best is trial 0 with value: 0.7324954052462568.
[I 2026-06-13 22:07:41,425] Trial 2 finished with value: 0.5898678696566171 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.012439367209907218, 'subsample': 0.762378215816119, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 2.1371407316372935

In [ ]:
for target in ['prcDiabetes', 'prcStroke']:
    print(f"\n{'='*60}")
    print(f"Training XGBoost + LLM embeddings for: {disease_names[target]}")

    X_train = train_df_emb[features_with_emb]
    y_train = train_df_emb[target]
    X_test = test_df_emb[features_with_emb]
    y_test = test_df_emb[target]

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_test, y_test), n_trials=30, show_progress_bar=False)

    best_params = study.best_params
    print(f"Best params: {best_params}")

    final_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, **best_params)
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    wape = weighted_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    bias = mean_bias(y_test, y_pred)

    print(f"Test performance: WAPE={wape:.4f}, R2={r2:.4f}, Bias={bias:.4f}")

    emb_results.append({
        'Disease': disease_names[target],
        'WAPE': wape,
        'R2': r2,
        'Bias': bias,
        'Model_Type': 'XGBoost + LLM Emb'
    })

In [1]:
emb_df = pd.DataFrame(emb_results)
print("\nXGBoost + LLM Embeddings Summary:")
emb_df

NameError: name 'pd' is not defined